# Notebook 11 — Audit DB, hooks, and the live `dashboard.md`

**Purpose:** Close the observability loop. NB 02–10 produced work but discarded
the receipts; NB 11 wires three audiences to three persistent surfaces:

- **`audit.db`** (SQLite, append-only) — `ingest_history`, `cost_records`,
  `audit_events`. Compliance, retros, dashboard's cost block all read here.
- **Shell hooks** (`<wiki>/.wiki/config.toml`) — gate or notify on
  `on_ingest_complete` / `on_lint_complete`. Blocking hooks fail the job.
- **`dashboard.md`** — Dataview-rendered live health view, plus one
  server-rendered cost table fed from `audit.db`.

**Exam relevance:** Architecture Patterns (observability), Responsible AI
(audit trails). **Design refs:** §13 (observability), §14 (hooks), §13.3
(dashboard). **Depends on:** NB 02 (ingest), NB 08 (lint), NB 09 (jobs queue),
NB 10 (cache + cost records).

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import json
import os
import shutil
import stat
import subprocess
import time
from datetime import date, datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table

console = Console()
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set in .env"

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

POC_WIKI = REPO / "notebooks" / "data" / "poc-wiki"
NB_TMP = Path("/tmp/marginalia-nb11")
WIKI_ROOT = NB_TMP / "wiki"
DOT_WIKI = WIKI_ROOT / ".wiki"
JOBS_DB = DOT_WIKI / "jobs.db"
AUDIT_DB = DOT_WIKI / "audit.db"
HOOKS_DIR = DOT_WIKI / "hooks"
CONFIG_TOML = DOT_WIKI / "config.toml"

# Fresh slate per run — copy poc-wiki, init both DBs.
if NB_TMP.exists():
    shutil.rmtree(NB_TMP)
shutil.copytree(POC_WIKI, WIKI_ROOT)
DOT_WIKI.mkdir(parents=True, exist_ok=True)
HOOKS_DIR.mkdir(parents=True, exist_ok=True)

console.print(f"[dim]wiki_root  = [cyan]{WIKI_ROOT}[/cyan][/dim]")
console.print(f"[dim]audit.db   = [cyan]{AUDIT_DB}[/cyan][/dim]")
console.print(f"[dim]jobs.db    = [cyan]{JOBS_DB}[/cyan][/dim]")
console.print(f"[dim]hooks_dir  = [cyan]{HOOKS_DIR}[/cyan][/dim]")

In [ ]:
from anthropic import Anthropic

from engine.audit import (
    AuditWriter,
    cost_summary,
    daily_cost_breakdown,
    events_by_type,
    generate_dashboard,
    init_db as init_audit_db,
    last_n_ingests,
)
from engine.hooks import HookConfig, HookDispatcher, load_hook_config
from engine.jobs import JobStatus, connect, count_by_status, enqueue, init_db as init_jobs_db, list_jobs, run_worker
from engine.jobs.dispatchers import WorkerCtx
from engine.models.wiki_config import MarginaliaConfig
from engine.utils.cost_tracker import CostRecord, record_attempt

# We need padding under purpose.md to clear Anthropic's L3 cache minimum
# (~2-3K tokens for Haiku/Sonnet 4.x) — same trick NB 10 used.
purpose_path = WIKI_ROOT / "purpose.md"
purpose_path.write_text(
    purpose_path.read_text()
    + "\n\n## Extended scope (padding for L3 demo)\n"
    + "\n".join(
        f"- Topic {i:03d}: stable scope text covering area {i % 7} "
        "with consistent terminology, structured discussion, "
        "and reference to recurring decisions and ongoing initiatives."
        for i in range(50)
    ),
    encoding="utf-8",
)

init_jobs_db(JOBS_DB)
init_audit_db(AUDIT_DB)
config = MarginaliaConfig.load(WIKI_ROOT)
client = Anthropic()

console.print(f"[bold]Audit DB tables:[/bold]")
conn = __import__("sqlite3").connect(str(AUDIT_DB))
try:
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    for row in rows:
        console.print(f"  [cyan]{row[0]}[/cyan]")
finally:
    conn.close()

## Part A — The audit DB

Three append-only tables, one DB file per wiki. `ingest_history` summarises
each ingest call; `cost_records` lands one row per LLM attempt (cache hits
included as `cached=1` rows with zero tokens); `audit_events` carries typed
events with JSON metadata. The schema is design §13.2 verbatim.

The dispatcher already writes these rows automatically when the worker is
constructed with `audit_writer=AuditWriter(...)` and an optional
`hook_dispatcher`. We start by exercising `AuditWriter` directly to show
the round-trip, then run a real ingest job through the queue to confirm
the dispatcher persists everything.

In [ ]:
# Round-trip every row type once, just to prove the writer's contract.
with AuditWriter(AUDIT_DB) as w:
    w.record_ingest(
        job_id="demo-job-1",
        source_ref="raw/demo.md",
        source_hash="0" * 64,
        page_paths=["sources/demo"],
        tokens_in=1000,
        tokens_out=200,
        cost_usd=0.005,
        duration_ms=1234,
    )
    w.record_cost(
        record_attempt(
            agent="ingest", model="claude-haiku-4-5",
            tokens_in=1000, tokens_out=200, cached=False, job_id="demo-job-1",
        )
    )
    w.record_cost(
        record_attempt(
            agent="ingest", model="claude-haiku-4-5",
            tokens_in=0, tokens_out=0, cached=True, job_id="demo-job-1",
        )
    )
    w.record_event(
        event_type="contradiction_found",
        metadata={"page_a": "k/decisions/A", "page_b": "k/decisions/B", "severity": "high"},
        job_id="demo-job-1",
    )

ingests = last_n_ingests(AUDIT_DB, limit=5)
events = events_by_type(AUDIT_DB)
console.print(f"[green]ingest_history rows:[/green] {len(ingests)}")
console.print(f"[green]audit_events rows:[/green]  {len(events)}")
console.print(f"  example event: {events[0]['event_type']} severity={events[0]['metadata']['severity']}")

In [ ]:
# Now run a real ingest job through the queue. The handler picks up
# audit_writer + hook_dispatcher from WorkerCtx and persists rows.

# Start with a clean audit DB so we can read the new rows in isolation.
shutil.rmtree(AUDIT_DB.parent / "audit-demo", ignore_errors=True)
DEMO_AUDIT = DOT_WIKI / "audit.db"
DEMO_AUDIT.unlink(missing_ok=True)
init_audit_db(DEMO_AUDIT)

audit_writer = AuditWriter(DEMO_AUDIT)
dispatcher = HookDispatcher(config.hook_config, audit_writer=audit_writer)

ctx = WorkerCtx(
    wiki_root=WIKI_ROOT,
    config=config,
    client=client,
    db_path=JOBS_DB,
    audit_writer=audit_writer,
    hook_dispatcher=dispatcher,
)

conn = connect(JOBS_DB)
try:
    job_id = enqueue(conn, "ingest", {"input": str(WIKI_ROOT / "raw" / "good_source.md")})
finally:
    conn.close()

console.print(f"[dim]enqueued ingest job: {job_id[:8]}…[/dim]")
counts = await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)
console.print(f"[green]worker drained:[/green] {counts}")

In [ ]:
# Inspect what landed in audit.db.
ingests = last_n_ingests(DEMO_AUDIT, limit=5)
costs = cost_summary(DEMO_AUDIT, days=1)

ingest_table = Table(title="ingest_history")
ingest_table.add_column("job", overflow="fold")
ingest_table.add_column("source", overflow="fold")
ingest_table.add_column("page", overflow="fold")
ingest_table.add_column("tokens", justify="right")
ingest_table.add_column("cost", justify="right")
ingest_table.add_column("ms", justify="right")
for row in ingests:
    ingest_table.add_row(
        row["job_id"][:8],
        row["source_ref"].split("/")[-1],
        row["page_paths"][0],
        f"{row['tokens_in']}/{row['tokens_out']}",
        f"${row['cost_usd']:.4f}",
        str(row["duration_ms"]),
    )
console.print(ingest_table)

cost_table = Table(title="cost_records (aggregated by model)")
cost_table.add_column("model", style="cyan")
cost_table.add_column("calls", justify="right")
cost_table.add_column("cached", justify="right")
cost_table.add_column("tokens_in", justify="right")
cost_table.add_column("cost", justify="right")
for r in costs:
    cost_table.add_row(
        r.model, str(r.calls), f"{r.cached_calls}/{r.calls}",
        str(r.tokens_in), f"${r.cost_usd:.4f}"
    )
console.print(cost_table)

In [ ]:
# Re-enqueue the SAME source — L1 cache should hit. The cache decorator
# emits a CostRecord(cached=True, tokens_in=0) so the audit DB still
# tracks the call (with $0 cost) for a complete attribution trail.
conn = connect(JOBS_DB)
try:
    job_id_2 = enqueue(conn, "ingest", {"input": str(WIKI_ROOT / "raw" / "good_source.md")})
finally:
    conn.close()

# (Note: NB 10's cache lives at <wiki>/.wiki/cache/. The handler doesn't
# wire that today, so the second call still pays for analyze. Cache-aware
# dispatch is a follow-up. The audit DB still gets a full row.)
await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)

ingests_after = last_n_ingests(DEMO_AUDIT, limit=5)
console.print(f"[green]ingest_history rows after 2nd run:[/green] {len(ingests_after)}")

In [ ]:
# Pure-SQL demo: "every ingest of a local_file source in the last hour"
import sqlite3

cutoff = (datetime.now(timezone.utc).timestamp() - 3600)
conn = sqlite3.connect(str(DEMO_AUDIT))
conn.row_factory = sqlite3.Row
try:
    rows = conn.execute(
        "SELECT source_ref, tokens_in + tokens_out AS total_tokens, cost_usd, duration_ms "
        "FROM ingest_history "
        "ORDER BY timestamp DESC"
    ).fetchall()
finally:
    conn.close()

q_table = Table(title="Recent ingests via raw SQL")
q_table.add_column("source", overflow="fold")
q_table.add_column("tokens", justify="right")
q_table.add_column("cost", justify="right")
q_table.add_column("ms", justify="right")
for r in rows:
    q_table.add_row(
        r["source_ref"].split("/")[-1],
        str(r["total_tokens"]),
        f"${r['cost_usd']:.4f}",
        str(r["duration_ms"]),
    )
console.print(q_table)

In [ ]:
# CLI smoke — render the same data through `marginalia audit history`
result = subprocess.run(
    ["uv", "run", "marginalia", "audit", "history", "--db", str(DEMO_AUDIT), "--limit", "5"],
    cwd=REPO, capture_output=True, text=True,
)
print(result.stdout)

In [ ]:
# --json variant — pipe-to-jq friendly. One JSON object per line.
result = subprocess.run(
    ["uv", "run", "marginalia", "audit", "history", "--db", str(DEMO_AUDIT), "--json"],
    cwd=REPO, capture_output=True, text=True,
)
for line in result.stdout.splitlines():
    if line.strip():
        parsed = json.loads(line)
        console.print(f"[dim]json:[/dim] {parsed['source_ref']} → {parsed['page_paths']} (${parsed['cost_usd']:.4f})")

In [ ]:
# Cost summary CLI.
result = subprocess.run(
    ["uv", "run", "marginalia", "audit", "cost", "--db", str(DEMO_AUDIT), "--days", "1"],
    cwd=REPO, capture_output=True, text=True,
)
print(result.stdout)

## Part B — Shell hooks

`<wiki>/.wiki/config.toml` declares hooks per lifecycle event.
`HookDispatcher` reads the config, fires the hook on the right event with
JSON context piped to stdin, and enforces blocking/timeout semantics.

We exercise three modes:
- **Non-blocking notify** — exits 0 (or non-zero, doesn't matter), ingest succeeds.
- **Blocking gate** — exits 1 → ingest job lands as `failed`.
- **Timeout** — sleeps past `timeout_s` → killed → treated as failure.

In [ ]:
# 1. Non-blocking notify hook — pretty-print the context.
notify_path = HOOKS_DIR / "notify.sh"
notify_path.write_text(
    '#!/usr/bin/env bash\n'
    'read -r CTX\n'
    'echo "[notify-hook] received: $CTX" > "%s"\n'
    'exit 0\n' % (HOOKS_DIR / "notify.log"),
    encoding="utf-8",
)
notify_path.chmod(notify_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

CONFIG_TOML.write_text(
    "[hooks.on_ingest_complete]\n"
    f'command   = "{notify_path}"\n'
    "blocking  = false\n"
    "timeout_s = 5\n",
    encoding="utf-8",
)

# Reload config — hook_config now populated.
config = MarginaliaConfig.load(WIKI_ROOT)
console.print(f"[bold]config.hook_config:[/bold] {config.hook_config}")

In [ ]:
# Re-build the dispatcher with the new config and run another ingest.
audit_writer.close()
audit_writer = AuditWriter(DEMO_AUDIT)
dispatcher = HookDispatcher(config.hook_config, audit_writer=audit_writer)

ctx = WorkerCtx(
    wiki_root=WIKI_ROOT, config=config, client=client, db_path=JOBS_DB,
    audit_writer=audit_writer, hook_dispatcher=dispatcher,
)

conn = connect(JOBS_DB)
try:
    job_id_hook = enqueue(conn, "ingest", {"input": str(WIKI_ROOT / "raw" / "ambiguous_source.md")})
finally:
    conn.close()

await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)

# Inspect the hook log file.
log = (HOOKS_DIR / "notify.log").read_text(encoding="utf-8")
console.print(f"[green]notify hook fired:[/green] log={len(log)} chars")
console.print(f"[dim]{log[:200]}…[/dim]")

# Hook fired but didn't fail anything — ingest succeeded.
conn = connect(JOBS_DB)
try:
    job = next(j for j in list_jobs(conn, limit=10) if j.id == job_id_hook)
finally:
    conn.close()
console.print(f"[green]job status:[/green] {job.status.value}")

In [ ]:
# 2. Blocking gate hook — rejects every ingest.
gate_path = HOOKS_DIR / "deny.sh"
gate_path.write_text(
    '#!/usr/bin/env bash\n'
    'read -r CTX\n'
    'echo "[deny-hook] rejecting ingest: $CTX" >&2\n'
    'exit 1\n',
    encoding="utf-8",
)
gate_path.chmod(gate_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

CONFIG_TOML.write_text(
    "[hooks.on_ingest_complete]\n"
    f'command   = "{gate_path}"\n'
    "blocking  = true\n"
    "timeout_s = 5\n",
    encoding="utf-8",
)
config = MarginaliaConfig.load(WIKI_ROOT)
audit_writer.close()
audit_writer = AuditWriter(DEMO_AUDIT)
dispatcher = HookDispatcher(config.hook_config, audit_writer=audit_writer)
ctx = WorkerCtx(
    wiki_root=WIKI_ROOT, config=config, client=client, db_path=JOBS_DB,
    audit_writer=audit_writer, hook_dispatcher=dispatcher,
)

conn = connect(JOBS_DB)
try:
    blocked_job_id = enqueue(conn, "ingest", {"input": str(WIKI_ROOT / "raw" / "good_source.md")},
                              max_attempts=1)
finally:
    conn.close()

await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)

conn = connect(JOBS_DB)
try:
    blocked_job = next(j for j in list_jobs(conn, limit=10) if j.id == blocked_job_id)
finally:
    conn.close()
console.print(f"[red]blocked job status:[/red] {blocked_job.status.value}")
console.print(f"[dim]error: {(blocked_job.error or '')[:120]}[/dim]")

# Confirm the hook_failed audit event landed.
hook_fails = events_by_type(DEMO_AUDIT, event_type="hook_failed")
console.print(f"[yellow]hook_failed audit events:[/yellow] {len(hook_fails)}")

In [ ]:
# 3. Timeout — sleep past timeout_s, get killed, treated as failure.
slow_path = HOOKS_DIR / "slow.sh"
slow_path.write_text(
    '#!/usr/bin/env bash\nread -r CTX\nsleep 60\n',
    encoding="utf-8",
)
slow_path.chmod(slow_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

CONFIG_TOML.write_text(
    "[hooks.on_ingest_complete]\n"
    f'command   = "{slow_path}"\n'
    "blocking  = false\n"
    "timeout_s = 1\n",
    encoding="utf-8",
)
config = MarginaliaConfig.load(WIKI_ROOT)
audit_writer.close()
audit_writer = AuditWriter(DEMO_AUDIT)
dispatcher = HookDispatcher(config.hook_config, audit_writer=audit_writer)

# Just call dispatcher.fire directly — no need to enqueue a real ingest.
t0 = time.perf_counter()
result = dispatcher.fire("on_ingest_complete", {"job_id": "demo-timeout"})
elapsed = time.perf_counter() - t0
console.print(f"[bold]timeout demo:[/bold] elapsed={elapsed:.2f}s, timed_out={result.timed_out}, exit_code={result.exit_code}")
assert result.timed_out

In [ ]:
# Lint hook: register on_lint_complete + a lint job + fire it.
lint_hook = HOOKS_DIR / "lint-notify.sh"
lint_hook.write_text(
    '#!/usr/bin/env bash\n'
    'read -r CTX\n'
    'echo "[lint-hook] $CTX" > "%s"\n'
    'exit 0\n' % (HOOKS_DIR / "lint.log"),
    encoding="utf-8",
)
lint_hook.chmod(lint_hook.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

CONFIG_TOML.write_text(
    "[hooks.on_lint_complete]\n"
    f'command   = "{lint_hook}"\n'
    "blocking  = false\n"
    "timeout_s = 5\n",
    encoding="utf-8",
)
config = MarginaliaConfig.load(WIKI_ROOT)
audit_writer.close()
audit_writer = AuditWriter(DEMO_AUDIT)
dispatcher = HookDispatcher(config.hook_config, audit_writer=audit_writer)
ctx = WorkerCtx(
    wiki_root=WIKI_ROOT, config=config, client=client, db_path=JOBS_DB,
    audit_writer=audit_writer, hook_dispatcher=dispatcher,
)

conn = connect(JOBS_DB)
try:
    lint_job_id = enqueue(conn, "lint", {"threshold_days": 14}, max_attempts=1)
finally:
    conn.close()

await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)

conn = connect(JOBS_DB)
try:
    lint_job = next(j for j in list_jobs(conn, limit=10) if j.id == lint_job_id)
finally:
    conn.close()

console.print(f"[bold]lint job:[/bold] {lint_job.status.value}, result={lint_job.result}")

if (HOOKS_DIR / "lint.log").exists():
    log = (HOOKS_DIR / "lint.log").read_text(encoding="utf-8")
    console.print(f"[green]lint hook fired:[/green] {log[:200]}…")

# Lint events from this run.
contradictions = events_by_type(DEMO_AUDIT, event_type="contradiction_found")
stale = events_by_type(DEMO_AUDIT, event_type="stale_detected")
console.print(f"[dim]lint emitted:[/dim] {len(contradictions)} contradiction(s), {len(stale)} stale")

## Part C — Live `dashboard.md`

The dashboard mixes Dataview blocks (rendered live by Obsidian, fed from
page frontmatter) with one server-rendered cost table fed from `audit.db`.
Run `generate_dashboard()` after a busy day to refresh.

Open `<wiki>/dashboard.md` in Obsidian to see the Dataview blocks render
live — that's the manual step the design assumes.

In [ ]:
# Generate a fresh dashboard from current state.
dashboard_path = WIKI_ROOT / "dashboard.md"
body = generate_dashboard(WIKI_ROOT, audit_db_path=DEMO_AUDIT, days=7)
dashboard_path.write_text(body, encoding="utf-8")

console.print(f"[green]dashboard.md:[/green] {dashboard_path} ({len(body)} chars)")
print(body[:1500])

In [ ]:
# Run another lint job so the dashboard's cost block has fresh numbers.
conn = connect(JOBS_DB)
try:
    enqueue(conn, "lint", {}, max_attempts=1)
finally:
    conn.close()
await run_worker(ctx, max_jobs=1, stop_when_drained=True, poll_interval=0.05)

# Re-generate.
body_v2 = generate_dashboard(WIKI_ROOT, audit_db_path=DEMO_AUDIT, days=7)
dashboard_path.write_text(body_v2, encoding="utf-8")

# Show the diff in the cost section only.
import difflib
lines_v1 = body.splitlines(keepends=True)
lines_v2 = body_v2.splitlines(keepends=True)
diff = "".join(difflib.unified_diff(lines_v1, lines_v2, n=1, lineterm=""))
console.print(f"[bold]dashboard diff after another lint job:[/bold]")
print(diff[:1500] if diff else "(no diff — same numbers)")

## Part D — Receipts + what to extract

Numbers from this run feed `engine/decisions/observability.md` —
same pattern as `engine/decisions/cache.md` from NB 10.

In [ ]:
from datetime import date as _date

# Snapshot live numbers BEFORE building receipts string.
final_ingests = last_n_ingests(DEMO_AUDIT, limit=100)
final_costs = cost_summary(DEMO_AUDIT, days=30)
final_events_all = events_by_type(DEMO_AUDIT, limit=500)
final_hook_failed = events_by_type(DEMO_AUDIT, event_type="hook_failed")
final_contradictions = events_by_type(DEMO_AUDIT, event_type="contradiction_found")

NB11_NUMBERS = {
    "ingest_rows": len(final_ingests),
    "cost_calls": sum(c.calls for c in final_costs),
    "cost_cached_calls": sum(c.cached_calls for c in final_costs),
    "cost_total_usd": sum(c.cost_usd for c in final_costs),
    "events_total": len(final_events_all),
    "events_hook_failed": len(final_hook_failed),
    "events_contradiction_found": len(final_contradictions),
    "models": [{"model": c.model, "calls": c.calls, "cost": c.cost_usd} for c in final_costs],
}

receipts = f"""# Observability receipts (audit DB, hooks, dashboard)

**Last verified:** {_date.today().isoformat()}
**Generated by:** `notebooks/11_audit_hooks_dashboard.ipynb`
**Design refs:** `docs/marginalia-design.md` §13 (observability),
§13.2 (audit DB schema), §13.3 (dashboard.md), §14 (hooks).

## Architecture

```
                                 ingest / lint / synthesis job
                                              │
                                              ▼
                       ┌────────────────────────────────────────┐
                       │  WorkerCtx                             │
                       │    audit_writer: AuditWriter | None    │
                       │    hook_dispatcher: HookDispatcher|None│
                       └────────────────────────────────────────┘
                                  │                    │
                  on_cost callback│                    │ fire hook
                                  ▼                    ▼
                       ┌──────────────────┐   ┌──────────────────┐
                       │ AuditWriter      │   │ HookDispatcher   │
                       │  ingest_history  │   │  blocking → fail │
                       │  cost_records    │   │  non-blocking →  │
                       │  audit_events    │   │    log to audit  │
                       └──────────────────┘   └──────────────────┘
                                  │                    │
                                  ▼                    │
                       ┌──────────────────┐            │
                       │  audit.db        │◄───────────┘ hook_failed
                       │  (sibling of     │              audit_events row
                       │   jobs.db)       │
                       └──────────────────┘
                                  │
                            served via
                                  │
                       ┌──────────┴───────────┬────────────────┐
                       ▼                      ▼                ▼
              marginalia audit         dashboard.md       NB 12+ retros
              {{history,cost,events}}      (cost block)      (raw SQL)
```

## Headline numbers (this run)

| Measurement | Value |
|---|---|
| ingest_history rows | {NB11_NUMBERS['ingest_rows']} |
| cost_records calls (incl. cached) | {NB11_NUMBERS['cost_calls']} |
| ...of which cached hits | {NB11_NUMBERS['cost_cached_calls']} |
| total cost (USD) | ${NB11_NUMBERS['cost_total_usd']:.4f} |
| audit_events rows (all types) | {NB11_NUMBERS['events_total']} |
| ...contradiction_found | {NB11_NUMBERS['events_contradiction_found']} |
| ...hook_failed | {NB11_NUMBERS['events_hook_failed']} |

## Per-model cost (this run)

| Model | Calls | Cost (USD) |
|---|---|---|
""" + "\n".join(
    f"| `{m['model']}` | {m['calls']} | ${m['cost']:.4f} |" for m in NB11_NUMBERS["models"]
) + """

## Design decisions

| Decision | Choice | Why |
|---|---|---|
| Cost-record threading | `on_cost` callback on agent functions | Keeps `analyze_source` / `synthesize_page` agnostic of the audit DB. Agents own *building* records; callers own *persisting* them. Notebook flows can pass `list.append`; tests pass `Mock()`; dispatchers pass `audit_writer.record_cost`. |
| Lint integration | `_handle_lint` job dispatcher | Lint becomes a real queue job with retry semantics, not a one-off function. Hooks fire from the handler after `lint_wiki()` returns. |
| `audit.db` location | `<wiki>/.wiki/audit.db`, sibling of `jobs.db` | Same WAL-mode pattern, same default-path resolution. Operators see one wiki = two SQLite files. |
| Hook config | `<wiki>/.wiki/config.toml` via `tomllib` | Zero-deps (Python 3.11+ built-in). `MarginaliaConfig.load()` reads it opportunistically; missing file → `hook_config=None` → no hooks. |
| Hook semantics | `blocking` flag (default false), `timeout_s` (default 30) | Blocking + non-zero exit → `HookFailure` raised → job fails, retries via the §7.5 ladder. Non-blocking → `audit_events.event_type='hook_failed'` row, job succeeds. |
| Dashboard render | Dataview blocks + one server-rendered SQL table | Dataview can't query SQLite, so cost (the only audit-fed section) is materialised at generation time. Re-run `generate_dashboard` after busy days. |

## CLI

```text
marginalia audit history [--limit N] [--json]    # last N ingests
marginalia audit cost    [--days N]  [--json]    # rolling cost by model
marginalia audit events  [--type T]  [--json]    # filter by event_type
```

`--json` emits one JSON object per line — friendly to `jq`, `head`, etc.

## CACHE_VERSION discipline (carry-over from NB 10)

Cache hits still emit `cost_records` rows with `cached=1, tokens_in=0`.
The audit DB tracks the call (so attribution stays complete) without
double-counting tokens. Joining `cost_records` on `cached=1` is how the
NB 12+ "savings from cache" report will work.

## Out of scope (deliberate)

- Audit DB GC / retention (append-only by design; ops verb later).
- The `engine.log` JSON-lines stream (§13.1) — debugging surface, not exam-relevant.
- The `log.md` markdown audit log (§13.1) — same reasoning.
- CI hooks / GitHub Actions (different scope from per-wiki lifecycle hooks).
- Cost gates (active enforcement) — `cost_gate_triggered` event type
  exists in the schema; firing logic is future work.
"""

receipts_path = REPO / "engine" / "decisions" / "observability.md"
receipts_path.write_text(receipts, encoding="utf-8")
console.print(f"[green]wrote receipts:[/green] {receipts_path.relative_to(REPO)}")
console.print(f"[dim]{len(receipts)} bytes, {receipts.count(chr(10))} lines[/dim]")

## What to extract

| Notebook artifact | Lands at |
|---|---|
| Audit DB DDL + connect/init | `engine/audit/db.py` |
| AuditWriter (record_ingest, record_cost, record_event) | `engine/audit/writer.py` |
| Query helpers (last_n_ingests, cost_summary, events_by_type, daily_cost_breakdown) | `engine/audit/queries.py` |
| `generate_dashboard` | `engine/audit/dashboard.py` |
| `marginalia audit {history,cost,events}` Typer subapp | `engine/cli/audit.py` (registered in `engine/cli/main.py`) |
| Hook + HookConfig + tomllib loader | `engine/hooks/config.py` |
| HookDispatcher (fire, blocking, timeout, expanduser) | `engine/hooks/dispatcher.py` |
| Example hooks (Slack notify, confidence gate) | `engine/hooks/examples/` |
| Dashboard starter template | `scripts/dashboard_template.md` |
| `on_cost` callback wiring on agents + cache decorators | `engine/agents/ingest/{analyze,synthesize}.py`, `engine/cache/decorators.py` |
| `_handle_lint` job dispatcher + WorkerCtx audit/hook fields | `engine/jobs/dispatchers.py` |
| `MarginaliaConfig.hook_config` field + config.toml load | `engine/models/wiki_config.py` |
| Tests | `tests/test_audit_*.py`, `tests/test_hooks_*.py`, `tests/test_jobs_lint_dispatcher.py` (35 cases) |
| Receipts | `engine/decisions/observability.md` |

**Out of scope:** distributed audit DB, JSON-lines `engine.log`, markdown
`log.md`, audit retention/GC, CI hooks. NB 11 lands the load-bearing
SQLite surface; the others are future work.